# Script 04 · Índices espectrales

**Curso:** Introducción a Google Earth Engine  
**Autora:** Grettel Vargas Azofeifa  
**Modalidad:** material de apoyo para GitHub y Google Earth Engine

> Los bloques de código están escritos en JavaScript para ejecutarse en el Editor de código de Google Earth Engine.

# Script 04 — Índices espectrales

Calcule y visualice índices espectrales utilizando mosaicos ligeros de Sentinel-2 y Landsat 9 que cubren completamente Guanacaste y Puntarenas.

## 🎯 Objetivos

1. Seleccionar una imagen Sentinel-2 poco nubosa.
2. Visualizar color natural, falso color y agricultura.
3. Calcular NDVI, EVI, NDWI y NBR.
4. Seleccionar una imagen Landsat 9 poco nubosa.
5. Visualizar color natural, falso color y agricultura con Landsat.
6. Calcular NDVI, EVI, NDWI y NBR con Landsat.
7. Comparar de forma visual ambos sensores.

> **Nota:** Enfoque: se usa un periodo corto y únicamente las bandas necesarias. No se calculan estadísticas ni histogramas.

## Información del ejercicio

## 2. Definir parámetros y área de interés

Las provincias, el periodo y la nubosidad máxima se establecen una sola vez al principio del código. Los mismos parámetros se reutilizan para Sentinel-2 y Landsat 9.

In [ ]:
// Parámetros generales.
var nombresProvincias = [
  'Guanacaste',
  'Puntarenas'
];

var fechaInicio = '2024-01-01';
var fechaFin = '2024-03-31';
var nubosidadMaxima = 30;

// Límites administrativos.
var provincias = ee.FeatureCollection(
  'FAO/GAUL/2015/level1'
).filter(
  ee.Filter.eq(
    'ADM0_NAME',
    'Costa Rica'
  )
);

// Seleccionar las dos provincias.
var provinciasSeleccionadas = provincias.filter(
  ee.Filter.inList(
    'ADM1_NAME',
    nombresProvincias
  )
);

// Crear una sola área de interés.
var areaInteres =
  provinciasSeleccionadas.geometry();

Map.centerObject(areaInteres, 8);
Map.setOptions('HYBRID');


> **Nota:** Ventaja: para cambiar el análisis solo debe modificar la lista de provincias, las fechas o la nubosidad máxima al inicio del script.

## 3. Cargar Sentinel-2

La colección utiliza el área, las fechas y la nubosidad definidos al inicio del script.

In [ ]:
var sentinel = ee.ImageCollection(
  'COPERNICUS/S2_SR_HARMONIZED'
)
  .filterBounds(areaInteres)
  .filterDate(
    fechaInicio,
    fechaFin
  )
  .filter(
    ee.Filter.lte(
      'CLOUDY_PIXEL_PERCENTAGE',
      nubosidadMaxima
    )
  )
  .select([
    'B2',
    'B3',
    'B4',
    'B8',
    'B11',
    'B12'
  ])
  .sort('CLOUDY_PIXEL_PERCENTAGE');

print(
  'Cantidad de imágenes Sentinel-2:',
  sentinel.size()
);

var imagenSentinel = sentinel
  .median()
  .clip(areaInteres);


## 4. Composiciones Sentinel-2

Se visualiza la misma imagen en color natural, falso color y agricultura.

In [ ]:
var sentinelNatural = {
  bands: ['B4', 'B3', 'B2'],
  min: 0,
  max: 3000
};

var sentinelFalsoColor = {
  bands: ['B8', 'B4', 'B3'],
  min: 0,
  max: 3000
};

var sentinelAgricultura = {
  bands: ['B11', 'B8', 'B2'],
  min: 0,
  max: 3000
};

Map.addLayer(
  imagenSentinel,
  sentinelNatural,
  'Sentinel-2 · Natural'
);

Map.addLayer(
  imagenSentinel,
  sentinelFalsoColor,
  'Sentinel-2 · Falso color',
  false
);

Map.addLayer(
  imagenSentinel,
  sentinelAgricultura,
  'Sentinel-2 · Agricultura',
  false
);


## 5. Calcular NDVI

In [ ]:
var ndviSentinel = imagenSentinel
  .normalizedDifference([
    'B8',
    'B4'
  ]);

Map.addLayer(
  ndviSentinel,
  {
    min: -1,
    max: 1,
    palette: [
      '8b4513',
      'ffff00',
      '7fff00',
      '006400'
    ]
  },
  'Sentinel-2 · NDVI',
  false
);


## 6. Calcular EVI

In [ ]:
var eviSentinel = imagenSentinel.expression(
  '2.5 * ((NIR - RED) / ' +
  '(NIR + 6 * RED - 7.5 * BLUE + 1))',
  {
    NIR: imagenSentinel.select('B8'),
    RED: imagenSentinel.select('B4'),
    BLUE: imagenSentinel.select('B2')
  }
);


## 7. Calcular NDWI

In [ ]:
var ndwiSentinel = imagenSentinel
  .normalizedDifference([
    'B3',
    'B8'
  ]);


## 8. Calcular NBR

In [ ]:
var nbrSentinel = imagenSentinel
  .normalizedDifference([
    'B8',
    'B12'
  ]);


## 9. Cargar Landsat 9

La colección utiliza el área, las fechas y la nubosidad definidos al inicio del script.

In [ ]:
function escalarLandsat(imagen) {
  var bandasOpticas = imagen
    .select('SR_B.')
    .multiply(0.0000275)
    .add(-0.2);

  return imagen.addBands(
    bandasOpticas,
    null,
    true
  );
}

var landsat = ee.ImageCollection(
  'LANDSAT/LC09/C02/T1_L2'
)
  .filterBounds(areaInteres)
  .filterDate(
    fechaInicio,
    fechaFin
  )
  .filter(
    ee.Filter.lte(
      'CLOUD_COVER',
      nubosidadMaxima
    )
  )
  .map(escalarLandsat)
  .select([
    'SR_B2',
    'SR_B3',
    'SR_B4',
    'SR_B5',
    'SR_B6',
    'SR_B7'
  ])
  .sort('CLOUD_COVER');

print(
  'Cantidad de imágenes Landsat 9:',
  landsat.size()
);

var imagenLandsat = landsat
  .median()
  .clip(areaInteres);


## 10. Composiciones Landsat 9

El mosaico Landsat se visualiza en color natural, falso color y una combinación útil para agricultura. Todas las capas quedan recortadas a Guanacaste y Puntarenas.

In [ ]:
var landsatNatural = {
  bands: [
    'SR_B4',
    'SR_B3',
    'SR_B2'
  ],
  min: 0,
  max: 0.3,
  gamma: 1.2
};

var landsatFalsoColor = {
  bands: [
    'SR_B5',
    'SR_B4',
    'SR_B3'
  ],
  min: 0,
  max: 0.4,
  gamma: 1.2
};

var landsatAgricultura = {
  bands: [
    'SR_B6',
    'SR_B5',
    'SR_B2'
  ],
  min: 0,
  max: 0.4,
  gamma: 1.15
};

Map.addLayer(
  imagenLandsat,
  landsatNatural,
  'Landsat 9 · Color natural',
  false
);

Map.addLayer(
  imagenLandsat,
  landsatFalsoColor,
  'Landsat 9 · Falso color',
  false
);

Map.addLayer(
  imagenLandsat,
  landsatAgricultura,
  'Landsat 9 · Agricultura',
  false
);


> **Práctica:** 🧪 Práctica: active las tres composiciones Landsat desde Layers y compárelas con las composiciones Sentinel-2.

## 11. Índices Landsat 9

Se calculan NDVI, EVI, NDWI y NBR con las bandas equivalentes de Landsat 9. Todos los resultados se recortan al área de interés.

In [ ]:
var ndviLandsat = imagenLandsat
  .normalizedDifference([
    'SR_B5',
    'SR_B4'
  ])
  .rename('NDVI_Landsat')
  .clip(areaInteres);

var eviLandsat = imagenLandsat.expression(
  '2.5 * ((NIR - RED) / ' +
  '(NIR + 6 * RED - 7.5 * BLUE + 1))',
  {
    NIR: imagenLandsat.select('SR_B5'),
    RED: imagenLandsat.select('SR_B4'),
    BLUE: imagenLandsat.select('SR_B2')
  }
)
  .rename('EVI_Landsat')
  .clip(areaInteres);

var ndwiLandsat = imagenLandsat
  .normalizedDifference([
    'SR_B3',
    'SR_B5'
  ])
  .rename('NDWI_Landsat')
  .clip(areaInteres);

var nbrLandsat = imagenLandsat
  .normalizedDifference([
    'SR_B5',
    'SR_B7'
  ])
  .rename('NBR_Landsat')
  .clip(areaInteres);

Map.addLayer(
  ndviLandsat,
  {
    min: -1,
    max: 1,
    palette: [
      '8b4513',
      'ffff00',
      '7fff00',
      '006400'
    ]
  },
  'Landsat 9 · NDVI',
  false
);

Map.addLayer(
  eviLandsat,
  {
    min: -1,
    max: 1,
    palette: [
      '8b4513',
      'ffff00',
      '7fff00',
      '006400'
    ]
  },
  'Landsat 9 · EVI',
  false
);

Map.addLayer(
  ndwiLandsat,
  {
    min: -1,
    max: 1,
    palette: [
      '8b4513',
      'ffffcc',
      '41b6c4',
      '225ea8'
    ]
  },
  'Landsat 9 · NDWI',
  false
);

Map.addLayer(
  nbrLandsat,
  {
    min: -1,
    max: 1,
    palette: [
      '7f0000',
      'fdae61',
      'ffffbf',
      'a6d96a',
      '1a9850'
    ]
  },
  'Landsat 9 · NBR',
  false
);


## 12. Comparar Sentinel-2 y Landsat

1. Active Sentinel-2 en color natural.
2. Active Landsat 9 en color natural.
3. Compare el nivel de detalle.
4. Repita la comparación con NDVI.

> **Nota:** Sentinel-2 ofrece mayor detalle espacial; Landsat permite analizar periodos históricos más extensos.

## 🚀 Desafío opcional

Cambie una de las provincias de la lista y repita el ejercicio.

## ✅ Resumen

- Seleccionamos una imagen Sentinel-2.
- Visualizamos tres composiciones.
- Calculamos NDVI, EVI, NDWI y NBR.
- Seleccionamos una imagen Landsat 9.
- Visualizamos Landsat en color natural, falso color y agricultura.
- Calculamos NDVI, EVI, NDWI y NBR con Landsat.
- Comparamos ambos sensores.

> **Nota:** Siguiente paso: en el Script 05 se exportarán los resultados.

## 💻 Código completo

In [ ]:
//==================================================
// INFORMACIÓN DEL EJERCICIO
//==================================================
//
// SENTINEL-2
// 13 bandas; resolución de 10, 20 y 60 m.
// Bandas utilizadas: B2, B3, B4, B8, B11 y B12.
//
// LANDSAT 9
// 11 bandas; resolución de 30 m en bandas ópticas.
// Bandas utilizadas:
// SR_B2, SR_B3, SR_B4, SR_B5, SR_B6 y SR_B7.
//
// Productos:
// Color natural, falso color, agricultura,
// NDVI, EVI, NDWI y NBR.
//
//==================================================


//==================================================
// PARÁMETROS GENERALES
//==================================================
var nombresProvincias = [
  'Guanacaste',
  'Puntarenas'
];

var fechaInicio = '2024-01-01';
var fechaFin = '2024-03-31';
var nubosidadMaxima = 30;

var provincias = ee.FeatureCollection(
  'FAO/GAUL/2015/level1'
).filter(
  ee.Filter.eq('ADM0_NAME', 'Costa Rica')
);

var provinciasSeleccionadas = provincias.filter(
  ee.Filter.inList(
    'ADM1_NAME',
    nombresProvincias
  )
);

var areaInteres = provinciasSeleccionadas.geometry();

Map.centerObject(areaInteres, 8);
Map.setOptions('HYBRID');

// 3. BUSCAR UNA IMAGEN SENTINEL-2
var sentinel = ee.ImageCollection(
  'COPERNICUS/S2_SR_HARMONIZED'
)
  .filterBounds(areaInteres)
  .filterDate(
    fechaInicio,
    fechaFin
  )
  .filter(
    ee.Filter.lte(
      'CLOUDY_PIXEL_PERCENTAGE',
      nubosidadMaxima
    )
  )
  .select([
    'B2',
    'B3',
    'B4',
    'B8',
    'B11',
    'B12'
  ])
  .sort(
    'CLOUDY_PIXEL_PERCENTAGE',
    false
  );

print(
  'Cantidad de imágenes Sentinel-2:',
  sentinel.size()
);

var imagenSentinel = sentinel
  .median()
  .clip(areaInteres);

// 4. COMPOSICIONES SENTINEL-2
var sentinelNatural = {
  bands: ['B4', 'B3', 'B2'],
  min: 0,
  max: 3000,
  gamma: 1.2
};

var sentinelFalsoColor = {
  bands: ['B8', 'B4', 'B3'],
  min: 0,
  max: 3000,
  gamma: 1.2
};

var sentinelAgricultura = {
  bands: ['B11', 'B8', 'B2'],
  min: 0,
  max: 3000,
  gamma: 1.15
};

Map.addLayer(
  imagenSentinel,
  sentinelNatural,
  'Sentinel-2 · Color natural'
);

Map.addLayer(
  imagenSentinel,
  sentinelFalsoColor,
  'Sentinel-2 · Falso color',
  false
);

Map.addLayer(
  imagenSentinel,
  sentinelAgricultura,
  'Sentinel-2 · Agricultura',
  false
);

// 5. NDVI SENTINEL-2
var ndviSentinel = imagenSentinel
  .normalizedDifference([
    'B8',
    'B4'
  ])
  .rename('NDVI');

Map.addLayer(
  ndviSentinel,
  {
    min: -1,
    max: 1,
    palette: [
      '8b4513',
      'ffff00',
      '7fff00',
      '006400'
    ]
  },
  'Sentinel-2 · NDVI',
  false
);

// 6. EVI SENTINEL-2
var eviSentinel = imagenSentinel.expression(
  '2.5 * ((NIR - RED) / ' +
  '(NIR + 6 * RED - 7.5 * BLUE + 1))',
  {
    NIR: imagenSentinel.select('B8'),
    RED: imagenSentinel.select('B4'),
    BLUE: imagenSentinel.select('B2')
  }
).rename('EVI');

Map.addLayer(
  eviSentinel,
  {
    min: -1,
    max: 1,
    palette: [
      '8b4513',
      'ffff00',
      '7fff00',
      '006400'
    ]
  },
  'Sentinel-2 · EVI',
  false
);

// 7. NDWI SENTINEL-2
var ndwiSentinel = imagenSentinel
  .normalizedDifference([
    'B3',
    'B8'
  ])
  .rename('NDWI');

Map.addLayer(
  ndwiSentinel,
  {
    min: -1,
    max: 1,
    palette: [
      '8b4513',
      'ffffcc',
      '41b6c4',
      '225ea8'
    ]
  },
  'Sentinel-2 · NDWI',
  false
);

// 8. NBR SENTINEL-2
var nbrSentinel = imagenSentinel
  .normalizedDifference([
    'B8',
    'B12'
  ])
  .rename('NBR');

Map.addLayer(
  nbrSentinel,
  {
    min: -1,
    max: 1,
    palette: [
      '7f0000',
      'fdae61',
      'ffffbf',
      'a6d96a',
      '1a9850'
    ]
  },
  'Sentinel-2 · NBR',
  false
);

// 9. BUSCAR UNA IMAGEN LANDSAT 9
function escalarLandsat(imagen) {
  var bandasOpticas = imagen
    .select('SR_B.')
    .multiply(0.0000275)
    .add(-0.2);

  return imagen.addBands(
    bandasOpticas,
    null,
    true
  );
}

var landsat = ee.ImageCollection(
  'LANDSAT/LC09/C02/T1_L2'
)
  .filterBounds(areaInteres)
  .filterDate(
    fechaInicio,
    fechaFin
  )
  .filter(
    ee.Filter.lte(
      'CLOUD_COVER',
      nubosidadMaxima
    )
  )
  .map(escalarLandsat)
  .select([
    'SR_B2',
    'SR_B3',
    'SR_B4',
    'SR_B5',
    'SR_B6',
    'SR_B7'
  ])
  .sort(
    'CLOUD_COVER',
    false
  );

print(
  'Cantidad de imágenes Landsat 9:',
  landsat.size()
);

var imagenLandsat = landsat
  .median()
  .clip(areaInteres);

// 10. COMPOSICIONES LANDSAT
var landsatNatural = {
  bands: ['SR_B4', 'SR_B3', 'SR_B2'],
  min: 0,
  max: 0.3,
  gamma: 1.2
};

var landsatFalsoColor = {
  bands: ['SR_B5', 'SR_B4', 'SR_B3'],
  min: 0,
  max: 0.4,
  gamma: 1.2
};

var landsatAgricultura = {
  bands: ['SR_B6', 'SR_B5', 'SR_B2'],
  min: 0,
  max: 0.4,
  gamma: 1.15
};

Map.addLayer(
  imagenLandsat,
  landsatNatural,
  'Landsat 9 · Color natural',
  false
);

Map.addLayer(
  imagenLandsat,
  landsatFalsoColor,
  'Landsat 9 · Falso color',
  false
);

Map.addLayer(
  imagenLandsat,
  landsatAgricultura,
  'Landsat 9 · Agricultura',
  false
);

// 11. ÍNDICES LANDSAT
var ndviLandsat = imagenLandsat
  .normalizedDifference(['SR_B5', 'SR_B4'])
  .rename('NDVI_Landsat')
  .clip(areaInteres);

var eviLandsat = imagenLandsat.expression(
  '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
  {
    NIR: imagenLandsat.select('SR_B5'),
    RED: imagenLandsat.select('SR_B4'),
    BLUE: imagenLandsat.select('SR_B2')
  }
)
  .rename('EVI_Landsat')
  .clip(areaInteres);

var ndwiLandsat = imagenLandsat
  .normalizedDifference(['SR_B3', 'SR_B5'])
  .rename('NDWI_Landsat')
  .clip(areaInteres);

var nbrLandsat = imagenLandsat
  .normalizedDifference(['SR_B5', 'SR_B7'])
  .rename('NBR_Landsat')
  .clip(areaInteres);

Map.addLayer(
  ndviLandsat,
  {
    min: -1,
    max: 1,
    palette: ['8b4513', 'ffff00', '7fff00', '006400']
  },
  'Landsat 9 · NDVI',
  false
);

Map.addLayer(
  eviLandsat,
  {
    min: -1,
    max: 1,
    palette: ['8b4513', 'ffff00', '7fff00', '006400']
  },
  'Landsat 9 · EVI',
  false
);

Map.addLayer(
  ndwiLandsat,
  {
    min: -1,
    max: 1,
    palette: ['8b4513', 'ffffcc', '41b6c4', '225ea8']
  },
  'Landsat 9 · NDWI',
  false
);

Map.addLayer(
  nbrLandsat,
  {
    min: -1,
    max: 1,
    palette: ['7f0000', 'fdae61', 'ffffbf', 'a6d96a', '1a9850']
  },
  'Landsat 9 · NBR',
  false
);

// 12. AÑADIR EL LÍMITE
Map.addLayer(
  provinciasSeleccionadas.style({
    color: 'ffffff',
    fillColor: '00000000',
    width: 3
  }),
  {},
  'Límite del área de interés'
);

// 13. RESUMEN
print(
  'Resumen:',
  'Se visualizaron composiciones Sentinel-2, se calcularon ' +
  'NDVI, EVI, NDWI y NBR, y se comparó NDVI con Landsat 9.'
);


## 📚 Recursos oficiales

- normalizedDifference()
- ee.Image.expression()
- Sentinel-2
- Landsat 9